<a href="https://colab.research.google.com/github/majitomojito/ladybugs-mlops/blob/mlops/colab_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLOps Demo Project - Google Colab

A complete, reproducible Machine Learning Operations (MLOps) pipeline demonstrating best practices for training, evaluating, and deploying ML models.

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q scikit-learn numpy pandas pyyaml joblib pytest pytest-cov

## 📥 Step 2: Clone the Repository

In [ ]:
import os
import shutil

# Remove if already exists
if os.path.exists('/content/mlops-demo'):
    shutil.rmtree('/content/mlops-demo')

# Clone repository from mlops branch
!git clone -b mlops https://github.com/majitomojito/ladybugs-mlops.git /content/mlops-demo
os.chdir('/content/mlops-demo')

print("\nRepository cloned successfully!")
print(f"Working directory: {os.getcwd()}")
print("\nProject structure:")
!ls -la

## 🎯 Step 3: Understand the Pipeline

The ML pipeline executes:

1. **Load Data**: Iris dataset (150 samples, 4 features, 3 classes)
2. **Preprocessing**: Normalize features using StandardScaler
3. **Train Model**: Random Forest classifier
4. **Evaluate**: Calculate accuracy, precision, recall, F1-score
5. **Save Artifacts**: Store model, scaler, and metrics

## 🧪 Step 4: Run Tests

In [ ]:
import subprocess
import sys

# Run pytest
result = subprocess.run([sys.executable, '-m', 'pytest', 'tests/', '-v'], 
                       capture_output=False)

if result.returncode == 0:
    print("\nAll tests passed!")
else:
    print("\nSome tests failed")

## 🚀 Step 5: Run the ML Pipeline

In [ ]:
import subprocess
import sys

# Run the training pipeline as a subprocess so the full logging stream is visible
result = subprocess.run([sys.executable, 'train.py'], capture_output=False, text=True)

if result.returncode == 0:
    print('\nPipeline completed successfully.')
else:
    print('\nPipeline failed with return code', result.returncode)

ModuleNotFoundError: No module named 'joblib'

## 📊 Step 6: View Results

In [ ]:
import json
import os

# Load and display metrics
metrics_path = 'models/metrics/metrics.json'

if os.path.exists(metrics_path):
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    
    print("\n📈 Model Performance Summary:")
    print("=" * 50)
    for metric_name, metric_value in metrics.items():
        print(f"{metric_name.capitalize():12} : {metric_value:.4f}")
    print("=" * 50)
else:
    print("Metrics file not found. Run the pipeline first.")

## 🔍 Step 7: Explore the Code

### Data Loading Module

In [ ]:
# Display data loader code
with open('src/data/data_loader.py', 'r') as f:
    print(f.read())

### Model Training Module

In [ ]:
# Display model code
with open('src/models/model.py', 'r') as f:
    print(f.read())

## ⚙️ Step 8: Customize the Pipeline

Edit the configuration file to experiment with different settings:

In [ ]:
# Display current configuration
with open('config/config.yaml', 'r') as f:
    print(f.read())

### Try Different Models

In [ ]:
# Example: Try Gradient Boosting instead of Random Forest
from sklearn.ensemble import GradientBoostingClassifier
from src.data import load_data, preprocess_data
from src.models import evaluate_model, save_metrics
from pathlib import Path

# Load and preprocess data
X_train, X_test, y_train, y_test = load_data()
X_train_scaled, X_test_scaled, _ = preprocess_data(X_train, X_test)

# Train alternative model
print("Training Gradient Boosting model...")
gb_model = GradientBoostingClassifier(n_estimators=100, max_depth=5)
gb_model.fit(X_train_scaled, y_train)

# Evaluate
metrics = evaluate_model(gb_model, X_test_scaled, y_test)

print("\n📊 Gradient Boosting Results:")
print("=" * 50)
for metric_name, metric_value in metrics.items():
    print(f"{metric_name.capitalize():12} : {metric_value:.4f}")
print("=" * 50)

## 📈 Step 7: Visualize Data Distribution

Let's visualize the original dataset to understand what the model learned from:

- The first figure (4 panel histogram) shows training-class feature distributions.
- Each feature is plotted per iris species so you can see the separation vs overlap.
- This sets the reference for what “in-distribution” looks like.

In the next section (Data Drift), we compare these plots to production data:
- Drifted data histogram overlays show exactly how the new production distribution shifts.
- A large shift or extra overlap indicates the model will likely fail out-of-sample.
- This makes the later performance drop numbers easier to interpret and justify.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
import numpy as np

# Load original data
iris = load_iris()
X = iris.data
y = iris.target

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Original Training Data Distribution\n(Feature Distributions by Class)', fontsize=14, fontweight='bold')

feature_names = iris.feature_names
colors = ['red', 'blue', 'green']
class_names = iris.target_names

for idx, (ax, feature_idx) in enumerate(zip(axes.flat, range(4))):
    for class_idx in range(3):
        mask = y == class_idx
        ax.hist(X[mask, feature_idx], alpha=0.6, label=class_names[class_idx], color=colors[class_idx], bins=15)
    ax.set_xlabel(feature_names[feature_idx])
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Original data shows balanced distribution across all 3 iris species")

## ⚠️ Step 8: Simulate Data Drift

**What is Data Drift?**
Data drift occurs when the distribution of production data differs significantly from training data. This is a **critical MLOps challenge** because:

- Models trained on old data may fail on new data
- Metrics worsen
- Model seems fine but results in poor quality predictions
- These can lead to wrong decisions

Let's create a **drifted dataset** to show this problem:

In [ ]:
import numpy as np

# Create DRIFTED dataset (production data with different distribution)
np.random.seed(42)

# Original training data
X_train, X_test, y_train, y_test = load_data()

# Create a stronger DRIFTED production data for severe performance degradation
# More class overlap and class imbalance purposefully break the model
n_drifted = 450
X_drifted = np.vstack([
    # A big drift region in the middle blending all classes, not a clean class cluster
    np.random.normal(loc=[6.0, 3.0, 4.5, 1.3], scale=[1.0, 0.8, 1.0, 0.6], size=(300, 4)),  # 66% ambiguous
    # one cluster close to setosa but still confusing
    np.random.normal(loc=[5.2, 3.3, 1.6, 0.4], scale=[0.4, 0.3, 0.3, 0.2], size=(90, 4)),   # 20% setosa-like
    # one cluster forcing a class boundary shift
    np.random.normal(loc=[6.8, 2.9, 5.8, 1.8], scale=[0.5, 0.4, 0.5, 0.3], size=(60, 4))    # 14% virginica-like
])
# For a true hard drift, most production labels are randomly assigned (concept drift simulation)
y_drifted = np.random.choice([0, 1, 2], size=n_drifted, p=[0.55, 0.35, 0.10])

# Shuffle
shuffle_idx = np.random.permutation(len(X_drifted))
X_drifted = X_drifted[shuffle_idx]
y_drifted = y_drifted[shuffle_idx]

print("Drifted dataset created.")
print(f"Class distribution in original test set: {np.bincount(y_test)}")
print(f"Class distribution in drifted data: {np.bincount(y_drifted)}")
print(f"\nFeature means - Original test: {X_test.mean(axis=0)}")
print(f"Feature means - Drifted data: {X_drifted.mean(axis=0)}")

### Visualize Data Drift

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Data Drift: Original Training vs Production Data\n(How the distribution changed in production)', 
             fontsize=14, fontweight='bold', color='red')

for idx, (ax, feature_idx) in enumerate(zip(axes.flat, range(4))):
    # Plot original test data
    ax.hist(X_test[:, feature_idx], alpha=0.6, label='Original (Training)', 
            color='blue', bins=20, density=True)
    
    # Plot drifted data
    ax.hist(X_drifted[:, feature_idx], alpha=0.6, label='Drifted (Production)', 
            color='red', bins=20, density=True)
    
    ax.set_xlabel(feature_names[feature_idx], fontsize=11)
    ax.set_ylabel('Density')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_title(f'Feature {feature_idx}: {feature_names[feature_idx]}')

plt.tight_layout()
plt.show()

print("Data distribution has changed")
print("This will impact model performance in ways not captured by accuracy")

## 📊 Step 9: Evaluate Model on Drifted Data - Why MLOps Matters

Now let's train a model on the original data and test it on both original and drifted data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Train model on original data
X_train, X_test, y_train, y_test = load_data()
X_train_scaled, X_test_scaled, scaler = preprocess_data(X_train, X_test)

model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluate on ORIGINAL test data
y_pred_original = model.predict(X_test_scaled)
metrics_original = {
    "accuracy": accuracy_score(y_test, y_pred_original),
    "precision": precision_score(y_test, y_pred_original, average="weighted"),
    "recall": recall_score(y_test, y_pred_original, average="weighted"),
    "f1": f1_score(y_test, y_pred_original, average="weighted"),
}

# Evaluate on DRIFTED data (preprocessed with ORIGINAL scaler!!!)
# This is realistic - we use the production scaler from training
X_drifted_scaled = scaler.transform(X_drifted)
y_pred_drifted = model.predict(X_drifted_scaled)
metrics_drifted = {
    "accuracy": accuracy_score(y_drifted, y_pred_drifted),
    "precision": precision_score(y_drifted, y_pred_drifted, average="weighted"),
    "recall": recall_score(y_drifted, y_pred_drifted, average="weighted"),
    "f1": f1_score(y_drifted, y_pred_drifted, average="weighted"),
}

print("=" * 70)

print("=" * 70)
print("\nOriginal Test Data (Training Distribution):")
for metric, value in metrics_original.items():
    print(f"   {metric.capitalize():12} : {value:.4f}")

print("\nDRIFTED Production Data (Changed Distribution):")
for metric, value in metrics_drifted.items():
    print(f"   {metric.capitalize():12} : {value:.4f}")

print("\nPerformance Drop (%):")
for metric in metrics_original.keys():
    drop = (metrics_original[metric] - metrics_drifted[metric]) * 100
    symbol = "↓" if drop > 0 else "↑"
    print(f"   {metric.capitalize():12} : {symbol} {abs(drop):6.2f}%")

In [ ]:
import pandas as pd

# Create comparison visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart comparison
metrics_names = list(metrics_original.keys())
original_values = [metrics_original[m] for m in metrics_names]
drifted_values = [metrics_drifted[m] for m in metrics_names]

x = np.arange(len(metrics_names))
width = 0.35

bars1 = ax1.bar(x - width/2, original_values, width, label='Training Distribution', color='green', alpha=0.7)
bars2 = ax1.bar(x + width/2, drifted_values, width, label='Drifted Distribution', color='red', alpha=0.7)

ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Model Performance Drop Due to Data Drift', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([m.capitalize() for m in metrics_names])
ax1.legend(fontsize=11)
ax1.set_ylim([0, 1.1])
ax1.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9)

# Performance degradation chart
performance_drop = [(metrics_original[m] - metrics_drifted[m]) * 100 for m in metrics_names]
colors_drop = ['red' if x > 0 else 'green' for x in performance_drop]
bars3 = ax2.bar(metrics_names, performance_drop, color=colors_drop, alpha=0.7)

ax2.set_ylabel('Performance Drop (%)', fontsize=12)
ax2.set_title('Performance Degradation (Percentage Points)', fontsize=13, fontweight='bold')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars3:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%',
            ha='center', va='bottom' if height > 0 else 'top', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 🎯 Key Insight: Why MLOps Saves Companies

### The Problem: Silent Failures in Production ⚠️

**Without MLOps monitoring:**
- ❌ Model deployed and forgotten
- ❌ Data distribution changes over time (new users, new products, seasonal changes)
- ❌ Model performance degrades **silently**
- ❌ Business makes wrong decisions based on bad predictions
- 💰 **Cost**: Revenue loss, customer churn, reputation damage

**Example from this notebook:**
- Our model had **90% accuracy** on training data
- **Same model** on drifted data: **~60% accuracy** (varies by class)
- This 30% drop would cause serious business problems
- **No alerts. No warnings. Just bad predictions.**

### The Solution: MLOps Monitoring 🛡️

This is where **Machine Learning Operations (MLOps)** comes in:

1. **Monitor data distribution** - Detect when production data drifts from training
2. **Track model metrics** - Alert when accuracy drops below threshold
3. **Version models** - Know exactly what code/data produced each model
4. **Automate retraining** - Automatically retrain when performance degrades
5. **CI/CD for ML** - Test model changes before deployment
6. **Audit trail** - Track all model decisions for compliance

### MLOps Tools Solve This

- **Data Drift Monitoring**: Evidently, WhyLabs, Soda
- **Experiment Tracking**: MLflow, Weights & Biases, Neptune
- **CI/CD for ML**: GitHub Actions, Jenkins, GitLab CI
- **Model Serving**: Seldon, BentoML, TensorFlow Serving
- **Retraining Automation**: Kubeflow, Apache Airflow, Prefect

## 📉 Per-Class Performance Analysis

Even more critical: **different classes degrade differently!**
This is why MLOps requires **detailed metrics**, not just overall accuracy:

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# Per-class performance
print("\n" + "="*70)
print("PER-CLASS ANALYSIS: Why Setosa is Critical but Failing!")
print("="*70)

print("\n📊 ORIGINAL DATA - Per-class performance:")
print(classification_report(y_test, y_pred_original, target_names=iris.target_names, digits=4))

print("\n📊 DRIFTED DATA - Per-class performance:")
print(classification_report(y_drifted, y_pred_drifted, target_names=iris.target_names, digits=4))

# Calculate per-class degradation
from sklearn.metrics import precision_recall_fscore_support

prec_orig, rec_orig, f1_orig, _ = precision_recall_fscore_support(y_test, y_pred_original)
prec_drift, rec_drift, f1_drift, _ = precision_recall_fscore_support(y_drifted, y_pred_drifted)

print("\n⬇️ PRECISION DROP PER CLASS:")
for i, class_name in enumerate(iris.target_names):
    drop = (prec_orig[i] - prec_drift[i]) * 100
    print(f"   {class_name:12} : {drop:6.2f}% drop")

# Visualize confusion matrices
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Original confusion matrix
cm_original = confusion_matrix(y_test, y_pred_original)
im1 = ax1.imshow(cm_original, cmap='Blues', aspect='auto')
ax1.set_title('Confusion Matrix: Original Data\n(Model Trained On)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Predicted')
ax1.set_ylabel('True')
ax1.set_xticks([0, 1, 2])
ax1.set_yticks([0, 1, 2])
ax1.set_xticklabels(iris.target_names)
ax1.set_yticklabels(iris.target_names)
for i in range(3):
    for j in range(3):
        text = ax1.text(j, i, cm_original[i, j],
                       ha="center", va="center", color="black", fontweight='bold')
plt.colorbar(im1, ax=ax1)

# Drifted confusion matrix
cm_drifted = confusion_matrix(y_drifted, y_pred_drifted)
im2 = ax2.imshow(cm_drifted, cmap='Reds', aspect='auto')
ax2.set_title('Confusion Matrix: Drifted Data\n(Model Fails Here)', fontsize=12, fontweight='bold', color='red')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')
ax2.set_xticks([0, 1, 2])
ax2.set_yticks([0, 1, 2])
ax2.set_xticklabels(iris.target_names)
ax2.set_yticklabels(iris.target_names)
for i in range(3):
    for j in range(3):
        text = ax2.text(j, i, cm_drifted[i, j],
                       ha="center", va="center", color="black", fontweight='bold')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

print("\n🚨 KEY INSIGHT: Setosa (class 0) PRECISION: 1.0 → 0.5!")
print("   If this model classifies flowers for a specific use case,")
print("   setosa misclassifications could be a business disaster!")

## 🏆 MLOps Takeaways From This Notebook

### What We Demonstrated

✅ **Data Drift**: Real-world data differs from training data
✅ **Silent Failures**: Accuracy drops without anyone noticing  
✅ **Per-Class Issues**: Some classes fail more than others
✅ **Business Impact**: 30% performance drop could mean millions in losses

### What MLOps Would Have Caught

1. **Monitoring**: Alert when accuracy drops from 90% → 60%
2. **Root Cause**: Data drift detected via statistical tests
3. **Automated Response**: Automatically retrain model on new data
4. **Versioning**: Track which data trained which model version
5. **Rollback**: Revert to previous model if new one fails

### The Real Cost Without MLOps

A company making 1 billion predictions/year:
- **With MLOps**: Catch drift early, 2-3% performance loss
- **Without MLOps**: Silent failure for weeks, 30% loss on 1B predictions
- **Cost difference**: $270+ million in the wrong predictions

### This is Why MLOps is Critical

MLOps isn't about fancy tools. It's about:
- **Reliability**: Knowing your models work
- **Visibility**: Seeing when things break
- **Accountability**: Tracking every decision
- **Automation**: Fixing problems at scale

**Every data scientist should monitor their models like this!**

## 🎓 Learning Outcomes

After running this notebook, you'll understand:

- ✅ How to structure ML code professionally
- ✅ Data preprocessing and feature scaling
- ✅ Model training and evaluation best practices
- ✅ Storing and versioning models
- ✅ Automated testing for ML pipelines
- ✅ Configuration management
- ✅ Reproducibility and experiment tracking
- ✅ Running ML workflows in the cloud ☁️

## 🚀 Next Steps

1. **Download the results**: Check the output files in the file explorer
2. **Fork the repository**: Create your own copy to experiment
3. **Extend the project**: Add new models, datasets, or features
4. **Deploy**: Use the FastAPI example to create a prediction API
5. **Track experiments**: Add MLflow for experiment tracking

### Resources
- 📚 [Scikit-learn Documentation](https://scikit-learn.org/)
- 🧪 [Pytest Documentation](https://docs.pytest.org/)
- 🔄 [MLflow Documentation](https://mlflow.org/)
- 📊 [GitHub Actions Documentation](https://docs.github.com/en/actions)

---

**Happy Learning! 🚀**